# ICOS atmosphere stations — five-segment sensitivity glyphs on a map

Reads **`icos_atmosphere_stations.csv`** and draws, at every station's `(lat, lon)`, a small
circle split into **five 72° segments** — one per column `trend`, `seasonal`, `synoptic`,
`diurnal`, `anthro` (in that order, clockwise from 12 o'clock). Each segment is filled with the
colour of its value on a **0 = red → ½ = white → 1 = blue** scale (`RdBu`), so a station that only
"sees" the long-term trend reads mostly red with one blue slice, while an urban/lowland site that
sees every signal reads mostly blue.

Basemap: Cartopy Natural-Earth land / ocean / coastlines / country borders, on a
Lambert-azimuthal-equal-area projection centred on Europe so the glyphs stay round across the
28 °N – 82 °N station spread.

Output → `./figures/fig_station_fingerprints.png`. Run from the repo root
(`c:\av\python\fluxnet`) so the CSV and `figures/` resolve next to it.

In [ ]:
# !pip install cartopy

import pathlib

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Wedge, Circle
from matplotlib.collections import PatchCollection

import cartopy.crs as ccrs
import cartopy.feature as cfeature

CSV = pathlib.Path("icos_atmosphere_stations.csv")

# Five glyph segments, in the order they are placed clockwise starting at 12 o'clock:
SEGMENTS   = ["trend", "seasonal", "synoptic", "diurnal", "anthro"]
SEG_LABELS = {"trend": "Trend", "seasonal": "Seasonal", "synoptic": "Synoptic",
              "diurnal": "Diurnal", "anthro": "Anthropogenic"}

CMAP        = plt.get_cmap("RdBu")              # value 0 -> red, 0.5 -> white, 1 -> blue
NORM        = mpl.colors.Normalize(0.0, 1.0)
SHOW_LABELS = True                              # print the station code under each glyph

OUTDIR = pathlib.Path("figures")
OUTDIR.mkdir(exist_ok=True)

df = pd.read_csv(CSV)
missing = [c for c in (["lat", "lon"] + SEGMENTS) if c not in df.columns]
assert not missing, f"CSV is missing required columns: {missing}"
# clip the segment values to [0, 1] just in case
for c in SEGMENTS:
    df[c] = df[c].clip(0.0, 1.0)

print(f"{len(df)} stations  |  lat {df.lat.min():.1f}–{df.lat.max():.1f} °N, "
      f"lon {df.lon.min():.1f}–{df.lon.max():.1f} °E")
df[["code", "name", "archetype", "lat", "lon"] + SEGMENTS].head(8)

## Build the map

Two panels share one colour scale: a **full view** of all stations (28 °N – 82 °N) and a **zoom**
on the busy NW/central-European cluster. `matplotlib`'s `Wedge` measures angles counter-clockwise
from east; segment *k* (0-based) is the *k*-th 72° slice going **clockwise from 12 o'clock**, i.e. it
spans `[90° − (k+1)·72°, 90° − k·72°]`. Wedge centres are the station lon/lat transformed into the
projection's coordinates; the glyph radius is a small fixed fraction of each panel's extent, so all
glyphs in a panel are the same size.

In [ ]:
PC     = ccrs.PlateCarree()
PROJ   = ccrs.LambertAzimuthalEqualArea(central_longitude=10.0, central_latitude=55.0)
DTHETA = 360.0 / len(SEGMENTS)

EXT_FULL = [df.lon.min() - 4.0, df.lon.max() + 4.0, df.lat.min() - 3.0, df.lat.max() + 3.0]
EXT_ZOOM = [-3.0, 22.0, 42.5, 61.0]                       # NW / central Europe


def add_basemap(ax, extent):
    ax.set_extent(extent, crs=PC)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"),     facecolor="#dbe7f1")
    ax.add_feature(cfeature.LAND.with_scale("50m"),      facecolor="#f3f0e7")
    ax.add_feature(cfeature.LAKES.with_scale("50m"),     facecolor="#dbe7f1", edgecolor="none")
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=0.7, edgecolor="#7d7d7d")
    ax.add_feature(cfeature.BORDERS.with_scale("50m"),   linewidth=0.4, edgecolor="#9b9b9b")
    gl = ax.gridlines(draw_labels=True, linewidth=0.4, color="0.8", alpha=0.6)
    gl.top_labels = gl.right_labels = False


def draw_glyphs(ax, frac, label_fs):
    """Draw a 5-segment glyph at every station that falls inside *ax*'s current extent.
    Returns the number of stations drawn."""
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    R = frac * max(x1 - x0, y1 - y0)
    seen, wedges, facecolors = [], [], []
    for _, row in df.iterrows():
        cx, cy = PROJ.transform_point(float(row.lon), float(row.lat), PC)
        if not (x0 <= cx <= x1 and y0 <= cy <= y1):
            continue
        seen.append((row, cx, cy))
        for k, seg in enumerate(SEGMENTS):
            wedges.append(Wedge((cx, cy), R, 90.0 - (k + 1) * DTHETA, 90.0 - k * DTHETA))
            facecolors.append(CMAP(NORM(float(row[seg]))))
    ax.add_collection(PatchCollection(wedges, facecolors=facecolors, edgecolor="white",
                                      linewidth=0.4, zorder=5))
    for row, cx, cy in seen:
        ax.add_patch(Circle((cx, cy), R, fill=False, edgecolor="0.2", linewidth=0.7, zorder=6))
        if label_fs and SHOW_LABELS:
            ax.text(cx, cy - 1.5 * R, str(row.code), ha="center", va="top", fontsize=label_fs,
                    color="0.12", zorder=7, clip_on=True)
    return len(seen)


def add_layout_key(ax, bounds):
    """A small reference pie (numbered wedges + legend) anchored at axes-fraction *bounds*."""
    key = ax.inset_axes(bounds)
    key.set_aspect("equal"); key.set_xlim(-1.6, 1.6); key.set_ylim(-3.6, 1.8)
    key.set_xticks([]); key.set_yticks([])
    for sp in key.spines.values():
        sp.set_visible(False)
    key.patch.set_facecolor("white"); key.patch.set_alpha(0.85); key.set_zorder(20)
    key.text(0, 1.6, "glyph layout", ha="center", va="bottom", fontsize=9, fontweight="bold")
    for k, seg in enumerate(SEGMENTS):
        t2, t1 = 90.0 - k * DTHETA, 90.0 - (k + 1) * DTHETA
        key.add_patch(Wedge((0, 0), 1.0, t1, t2, facecolor="0.82", edgecolor="white", linewidth=1.2))
        mid = np.deg2rad((t1 + t2) / 2.0)
        key.text(0.6 * np.cos(mid), 0.6 * np.sin(mid), str(k + 1), ha="center", va="center",
                 fontsize=9, fontweight="bold", color="0.25")
    key.add_patch(Circle((0, 0), 1.0, fill=False, edgecolor="0.3", linewidth=1.2))
    key.text(0, -1.45, "\n".join(f"{i + 1}   {SEG_LABELS[s]}" for i, s in enumerate(SEGMENTS)),
             ha="center", va="top", fontsize=8, linespacing=1.5)


fig = plt.figure(figsize=(17, 12.5))
gs  = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.08], wspace=0.05)
ax_full = fig.add_subplot(gs[0, 0], projection=PROJ)
ax_zoom = fig.add_subplot(gs[0, 1], projection=PROJ)

add_basemap(ax_full, EXT_FULL)
n_full = draw_glyphs(ax_full, frac=0.015, label_fs=6.0)
ax_full.set_title(f"all {n_full} stations", fontsize=12, pad=6)
add_layout_key(ax_full, [0.012, 0.40, 0.185, 0.215])
# outline of the zoom region on the full map
rl = [EXT_ZOOM[0], EXT_ZOOM[1], EXT_ZOOM[1], EXT_ZOOM[0], EXT_ZOOM[0]]
rb = [EXT_ZOOM[2], EXT_ZOOM[2], EXT_ZOOM[3], EXT_ZOOM[3], EXT_ZOOM[2]]
ax_full.plot(rl, rb, transform=PC, color="#444444", lw=1.2, ls=(0, (5, 3)), zorder=8)

add_basemap(ax_zoom, EXT_ZOOM)
n_zoom = draw_glyphs(ax_zoom, frac=0.040, label_fs=8.5)
ax_zoom.set_title(f"NW / central Europe — {n_zoom} stations (zoom)", fontsize=12, pad=6)

# shared colour-scale colourbar under both panels
sm = mpl.cm.ScalarMappable(norm=NORM, cmap=CMAP); sm.set_array([])
cbar = fig.colorbar(sm, ax=[ax_full, ax_zoom], orientation="horizontal", fraction=0.04,
                    pad=0.05, shrink=0.42, aspect=36)
cbar.set_ticks([0.0, 0.25, 0.5, 0.75, 1.0])
cbar.set_label("segment value        red = 0      ·      white = 0.5      ·      blue = 1")

fig.suptitle("ICOS atmosphere stations — five-segment sensitivity glyphs   "
             "(clockwise from top:  Trend · Seasonal · Synoptic · Diurnal · Anthropogenic;  "
             "each segment 0 = red → 1 = blue)", fontsize=13.5, y=0.97)

fig.savefig(OUTDIR / "fig_station_fingerprints.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
try:
    from IPython.display import Image, display
    _can_display = True
except ImportError:
    _can_display = False

p = OUTDIR / "fig_station_fingerprints.png"
print("Saved:", p.resolve())
if _can_display:
    display(Image(filename=str(p)))